In [ ]:
##### 해당 파일은 로컬환경에서 작업하였습니다 #####

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

import ujson as json
import glob
import pandas as pd
import numpy as np
import shutil
import os

In [ ]:
label_folder = []
first_path = "자유대화 음성(노인남녀)/*"
train_val_folder = glob.glob(first_path)
print(train_val_folder)

# 라벨 폴더만 선택
label_folder = []
for i in train_val_folder:
    i = i.replace('\\','/')
    label_folder_path = glob.escape(i)
    temp_folder = glob.glob(label_folder_path+"/*")
    for temp_ in temp_folder:
        if "zip" in temp_ or "원천" in temp_:
            continue
        label_folder.append(temp_)
print(label_folder)

json_list = []
for i in label_folder:
    i = i.replace('\\','/')
    json_path = glob.escape(i)
    temp_json_list = glob.glob(json_path+"/*/*.json")
    json_list.extend(temp_json_list)
print(len(json_list))

In [5]:
CLAP_A_3 = ['나비','피아노','바람개비','뜨거운','여름','바다같이','넓은','마음','남의','떡이','커','보인다','버스를','타고','집에','갔다','비','오는','날에는','우산을','쓴다','냉장고에','잘','익은','배가','두','개','있다','주말에','친구들과','기차를','타고','대구에','갔다']
CLAP_A_4 = ['덥다','덥고','더워','덥지','작다','작고','자가','작지','곱다','곱고','고와','곱지','달다','달고','달아','달지','지우다','지우고','지워','지우지','지운다']
CLAP_A_5 = ['모자','포도','해바라기','태극기','나무','무지개','기차','비행기','그네','신발']
CLAP_A_6 = ['소','돼지','닭','개','강아지','고양이','염소','오리','거위','당나귀','말','호랑이','토끼','여우','까치','거북이','용','뱀','개구리','쥐','원숭이','사슴','코끼리','사자','기린','얼룩말','캥거루','하마','코뿔소','타조','펭귄','북극곰','고릴라','제비','참새','비둘기','참치','멧돼지','두꺼비','메뚜기','잠자리','나비','벌','사마귀']
CLAP_A_7 = ['책가방','아이','현관','신발','강아지','꼬리','엄마','소파','뜨개질','아빠','신문','텔레비전','티비','테이블','로봇','라디오','창문','나무','참새','책장','책','달력','시계','가족사진','강아지집','밥그릇','식탁','주전자','찻잔','컵','과일','액자']

CLAP_D_1 = ['퍼','터','커']
CLAP_D_2 = ['나비','포도','기타','수갑','가위','모기','칼','꽁치','돼지','택시','호미','비행기','레몬','주사기','짬뽕','빨대','바퀴','쓰레기통','옷걸이','책장','떡국','바구니','열쇠','방패','얼음']
CLAP_D_3 = ['우리나라','싱그럽다','참새','짹잭','지저귀','푸른','새싹','꿈틀','수영장','물놀이','첨벙첨벙','물장구','치며','장난치','장난친']

total_list = list(set(CLAP_A_3 + CLAP_A_4 + CLAP_A_5 + CLAP_A_6 + CLAP_A_7 + CLAP_D_1 + CLAP_D_2 + CLAP_D_3))

In [ ]:
from concurrent.futures import ThreadPoolExecutor


# 파일명, 스크립트, 녹음시간 추출
def parse_json(json_file):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        info = data['발화정보']
        for word in total_list:
            if word in info['stt'].strip() and float(info['recrdTime']) <= 10:
                return {
                    'name': info['fileNm'],
                    'script': info['stt'].strip(),
                    'time': info['recrdTime']
                }
    # return None

# 병렬작업
with ThreadPoolExecutor() as executor:
    transcript_list = list(tqdm(executor.map(parse_json, json_list), total=len(json_list)))

In [ ]:
transcript_dict = {
    "name": [r['name'] for r in transcript_list if r is not None],
    "script": [r['script'] for r in transcript_list if r is not None],
    "time": [r['time'] for r in transcript_list if r is not None]
}

In [ ]:
# 최초 파일 리스트 저장
transcript_df = pd.DataFrame(transcript_dict)
transcript_df.to_csv("transcript.csv", index=False, encoding='utf-8-sig')

In [ ]:
# 녹음시간 형식정리
transcript_df = pd.read_csv('transcript.csv')
transcript_df['time'] = transcript_df['time'].astype(float)

In [ ]:
# 2초미만, 10초이상, 특수 토큰 포함 문장 drop
transcript_df.drop(transcript_df.loc[(transcript_df['time']<2) | (transcript_df['time']>=10) | (transcript_df['script'].str.contains('\\('))].index,inplace=True)
transcript_df.reset_index(drop=True, inplace=True)
transcript_df.to_csv('transcript_fin.csv', index=False, encoding='utf-8-sig')

In [ ]:
transcript_fin_df = pd.read_csv('transcript_fin.csv')
transcript_fin_df

,name,script,time
0,노인남여_노인대화02_F_e7222879_61_수도권_실내_01402.wavp,나 때문에 오늘 아침부터 마음 고생이 많아요 좀 멀리서,7.28
1,노인남여_노인대화02_F_e7222879_61_수도권_실내_01403.wavp,오느라고 전철을 하나 놓치는 바람에 그 다음부터 계속,6.84
2,노인남여_노인대화02_F_e7222879_61_수도권_실내_01404.wavp,늦어져 가지고 정말 너무 많이 미안하네요,6.29
3,노인남여_노인대화02_F_e7222879_61_수도권_실내_01408.wavp,언니가 아침부터 뛰느라고 정신 없었을 거 같아요,6.65
4,노인남여_노인대화02_F_e7222879_61_수도권_실내_01411.wavp,불안해 가지고 그래서 이것도 약속인데 아침부터 너무 좀,8.11
...,...,...,...
141934,노인남여_자유대화_M_1540293395_67_강원_실내_20201210062008...,캠핑 카의 밧데리를 인산 철 배터리로 교체하자,6.91
141935,노인남여_자유대화_M_1540293395_67_강원_실내_20201210062138...,언제든지 캠핑장에 가지 않고 세워 두면서 캠핑할 수 있는 여건을 만들자,8.70
141936,노인남여_자유대화_M_1540293395_67_강원_실내_20201210062220...,이렇게 준비해서 준비가 되는 대로 곧 여행을 떠날 생각입니다,6.91
141937,노인남여_자유대화_M_1540293395_67_강원_실내_20201210062343...,첫 번째는 건강 두 번째는 화목한 가정,6.06


In [ ]:
# 데이터 저장 폴더로 파일 이동
for idx in tqdm(range(len(transcript_df))):
    file_name = transcript_df.loc[idx,'name'].split('_')[-1].split('.')[0]
    folder_name = transcript_df.loc[idx,'name'].split(f'_{file_name}')[0]
    train_wav_path = f"D:/자유대화 음성(노인남녀)/Training/원천/{folder_name}/{folder_name}_{file_name}.wav"
    tarin_json_path = f"D:/자유대화 음성(노인남녀)/Training/라벨/{folder_name}/{folder_name}_{file_name}.json"
    valid_wav_path = f"D:/자유대화 음성(노인남녀)/Validation/{folder_name}/{folder_name}_{file_name}.wav"
    valid_json_path = f"D:/자유대화 음성(노인남녀)/Validation/{folder_name}/{folder_name}_{file_name}.json"
    pick_path = "D:/pick"

    if os.path.exists(f"D:/자유대화 음성(노인남녀)/Training/라벨/{folder_name}/{folder_name}_{file_name}.json") and os.path.exists(f"D:/자유대화 음성(노인남녀)/Training/원천/{folder_name}/{folder_name}_{file_name}.wav"):
        shutil.move(train_wav_path,pick_path)
        shutil.move(tarin_json_path,pick_path)
    elif os.path.exists(f"D:/자유대화 음성(노인남녀)/Validation/{folder_name}/{folder_name}_{file_name}.json") and os.path.exists(f"D:/자유대화 음성(노인남녀)/Validation/{folder_name}/{folder_name}_{file_name}.wav"):
        shutil.move(valid_wav_path,pick_path)
        shutil.move(valid_json_path,pick_path)
    else:
        print(file_name)

In [ ]:
len(glob.glob("D:/pick/*.wav"))

141939